# Notebook 1 — compute entropies in parallel batches, periodic Hamiltonian

This version uses a **periodic Hamiltonian** with a finite MPS representation:

```python
bc_MPS = "finite"
bc_x = "periodic"
```

The disorder-resolved DMRG entropies are computed for the fully critical interacting disordered Kitaev/Majorana chain. Batches are sequential; inside each batch disorder realizations are parallel. After every batch, all saved entropies and the updated average are written to one `.npz` file.

For the central-charge diagnostic this notebook uses the **PBC CFT form**

\[
S(\ell)=\frac{c}{3}\log\left[\frac{L}{\pi}\sin\left(\frac{\pi\ell}{L}\right)\right]+s_0.
\]

Since the fit variable includes the factor \(1/3\), the fitted slope is directly \(c\).


In [3]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Tuple
import traceback, time
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from tenpy.algorithms import dmrg
from tenpy.models.model import CouplingMPOModel
from tenpy.networks.mps import MPS
from tenpy.networks.site import FermionSite

plt.rcParams["figure.dpi"] = 200
plt.rc("text", usetex=False)
plt.rc("font", family="serif", size=12)

In [ ]:
# ============================================================
# Global parameters
# ============================================================

output_dir = Path("entropy_batches_fully_critical")
output_dir.mkdir(parents=True, exist_ok=True)

# Boundary condition:
# Use a periodic Hamiltonian, but keep a finite MPS.
# Do NOT set bc_MPS="periodic" unless you specifically want periodic-MPS DMRG.
bc_MPS = "finite"
bc_x = "periodic"
boundary_tag = "PBC"

main_entropy_file = output_dir / f"points_localization3_entropy_batches_fully_critical_{boundary_tag}_chi_100.npz"

batch_dir = output_dir / f"batches_{boundary_tag}"
batch_dir.mkdir(parents=True, exist_ok=True)

# Fully critical chain parameters
L = 250
t = 0.5
Delta = 0.5
mu = -(t + Delta)
W = 0.4
g = 0.5            # U = 2*g

# DMRG parameters
chi_max = 100
max_sweeps = 20
parity = "even"
svd_min = 1e-10
mixer = True
verbose_dmrg = 0

# Disorder batching
seed0 = 12345
batch_size = 20       # 10--20 as requested
n_batches = 50        # sequence of batches

# There are only batch_size tasks in one batch, so n_jobs > batch_size gives no benefit.
# For PBC DMRG, memory cost is higher than OBC. Start from 10--20 workers.
n_jobs = batch_size

resume = True

# Diagnostic fit only for progress output. Final fit is done in Notebook 2.
ell_min_diag = 5
ell_max_diag = min(18, L // 2)

ell_grid = np.arange(1, L // 2 + 1, dtype=int)

print("Boundary condition:", boundary_tag)
print("bc_MPS =", bc_MPS)
print("bc_x   =", bc_x)
print("Total planned new realizations:", batch_size * n_batches)
print("Parallel jobs per batch:", n_jobs)
print("Output:", main_entropy_file.resolve())


Boundary condition: PBC
bc_MPS = finite
bc_x   = periodic
Total planned new realizations: 1000
Parallel jobs per batch: 20
Output: /home/ubuntu/shapiro_projects/Majorana_disorder_KIT/entropy_batches_fully_critical/lifshitz4_entropy_batches_fully_critical_PBC_chi_100.npz


In [5]:
@dataclass
class DMRGResult:
    E: float
    n: np.ndarray
    entanglement: np.ndarray
    central_charge_fit: float
    cdw_q: float
    nk_abs: np.ndarray

class FullyCriticalDisorderedKitaevBDI(CouplingMPOModel):
    """
    Fully critical interacting disordered Kitaev/Majorana chain.

    The Hamiltonian can be open or periodic depending on model_params["bc_x"].

    For PBC use:
        bc_MPS = "finite"
        bc_x   = "periodic"

    The MPS remains finite, while the Hamiltonian contains the boundary terms.
    """

    def init_sites(self, model_params):
        return FermionSite(
            conserve=model_params.get("conserve", "parity"),
            filling=0.5,
        )

    def init_terms(self, model_params):
        L = int(model_params["L"])
        bc_x_local = str(model_params.get("bc_x", "open"))
        periodic = bc_x_local == "periodic"

        mu_array = np.asarray(model_params["mu_array"], dtype=float)
        t_array = np.asarray(model_params["t_array"], dtype=float)
        Delta_array = np.asarray(model_params["Delta_array"], dtype=float)
        U = float(model_params.get("U", 0.0))

        expected_bonds = L if periodic else L - 1

        if len(mu_array) != L:
            raise ValueError(f"mu_array must have length L={L}")

        if len(t_array) != expected_bonds:
            raise ValueError(
                f"t_array must have length {expected_bonds} for bc_x={bc_x_local}"
            )

        if len(Delta_array) != expected_bonds:
            raise ValueError(
                f"Delta_array must have length {expected_bonds} for bc_x={bc_x_local}"
            )

        # -mu_j (n_j - 1/2), constant omitted.
        self.add_onsite(-mu_array, 0, "N")

        # For bc_x="periodic", self.lat.pairs["nearest_neighbors"] includes
        # the boundary bond. Then t_array and Delta_array should have length L.
        for u1, u2, dx in self.lat.pairs["nearest_neighbors"]:
            self.add_coupling(-t_array, u1, "Cd", u2, "C", dx, plus_hc=True)
            self.add_coupling(Delta_array, u1, "C", u2, "C", dx, plus_hc=True)

            if abs(U) > 0.0:
                self.add_coupling(U, u1, "N", u2, "N", dx)

        # Linear terms from U(n_j - 1/2)(n_{j+1} - 1/2).
        if abs(U) > 0.0:
            U_linear = np.zeros(L, dtype=float)

            if periodic:
                # On a ring, each site belongs to two density-density bonds.
                U_linear[:] += -U
            else:
                # OBC: edge sites have one neighbor, bulk sites have two.
                U_linear[:-1] += -0.5 * U
                U_linear[1:] += -0.5 * U

            self.add_onsite(U_linear, 0, "N")

        # Uniform-Majorana three-site term with V=U:
        # -U/2 (Cd_j - C_j)(N_{j+1}-1/2)(Cd_{j+2}+C_{j+2})
        if abs(U) > 0.0:
            left_terms = [("Cd", +1.0), ("C", -1.0)]
            middle_terms = [("N", +1.0), ("Id", -0.5)]
            right_terms = [("Cd", +1.0), ("C", +1.0)]
            prefactor = -0.5 * U

            for opL, coefL in left_terms:
                for opM, coefM in middle_terms:
                    for opR, coefR in right_terms:
                        self.add_multi_coupling(
                            prefactor * coefL * coefM * coefR,
                            [(opL, [0], 0), (opM, [1], 0), (opR, [2], 0)],
                            op_string=None,
                            category="uniform_majorana_interaction",
                            plus_hc=False,
                        )


In [6]:
def make_fully_critical_disorder(
    L,
    *,
    t=0.5,
    Delta=0.5,
    mu=None,
    W=0.0,
    seed=None,
    bc_x="open",
):
    """
    Disorder convention:

        mu_j    = mu - W xi_j
        t_j     = t + W zeta_j / 2
        Delta_j = Delta + W zeta_j / 2

    For OBC, there are L-1 bonds.
    For PBC, there are L bonds, including the boundary bond.
    """

    if mu is None:
        mu = -(t + Delta)

    rng = np.random.default_rng(seed)

    periodic = str(bc_x) == "periodic"
    n_bonds = L if periodic else L - 1

    xi = rng.uniform(-1.0, 1.0, size=L)
    zeta = rng.uniform(-1.0, 1.0, size=n_bonds)

    return {
        "mu_array": mu - W * xi,
        "t_array": t + 0.5 * W * zeta,
        "Delta_array": Delta + 0.5 * W * zeta,
    }


def initial_product_state(L, parity="even"):
    state = ["empty", "full"] * (L // 2)

    if L % 2:
        state.append("empty")

    if parity == "odd":
        state[-1] = "full" if state[-1] == "empty" else "empty"
    elif parity != "even":
        raise ValueError("parity must be 'even' or 'odd'")

    return state


def x_cft_entropy(ell, L, bc_x="open"):
    """
    CFT coordinate used for extracting c.

    OBC:
        S(ell) = c/6 log[(2L/pi) sin(pi ell/L)] + const

    PBC:
        S(ell) = c/3 log[(L/pi) sin(pi ell/L)] + const

    Since the prefactor is included in x, the fitted slope is c.
    """

    ell = np.asarray(ell, dtype=float)

    if str(bc_x) == "periodic":
        return (1.0 / 3.0) * np.log(
            (L / np.pi) * np.sin(np.pi * ell / L)
        )

    return (1.0 / 6.0) * np.log(
        (2.0 * L / np.pi) * np.sin(np.pi * ell / L)
    )


def fit_c_from_S(S_half, L, ell_min, ell_max, bc_x="open"):
    S_half = np.asarray(S_half, dtype=float)

    ell = np.arange(1, len(S_half) + 1, dtype=int)
    ell_max_eff = min(int(ell_max), len(S_half))

    mask = (
        (ell >= int(ell_min))
        & (ell <= ell_max_eff)
        & np.isfinite(S_half)
    )

    if np.count_nonzero(mask) < 2:
        return np.nan, np.nan

    x = x_cft_entropy(ell[mask], L, bc_x=bc_x)
    y = S_half[mask]

    good = np.isfinite(x) & np.isfinite(y)

    if np.count_nonzero(good) < 2:
        return np.nan, np.nan

    c, s0 = np.polyfit(x[good], y[good], 1)

    return float(c), float(s0)


def cdw_q_from_density(n):
    n0 = np.asarray(n, dtype=float) - float(np.mean(n))
    nk = np.fft.rfft(n0)
    p2 = np.abs(nk[1:]) ** 2
    denom = float(np.sum(p2))

    q = 0.0 if denom == 0.0 else 2.0 * float(np.sum(p2 ** 2)) / denom ** 2

    return q, np.abs(nk)


In [7]:
def run_one_dmrg(
    *,
    L,
    t,
    Delta,
    mu,
    W,
    g,
    seed,
    parity,
    chi_max,
    max_sweeps,
    bc_MPS="finite",
    bc_x="open",
    svd_min=1e-10,
    mixer=True,
    verbose=0,
    ell_min_diag=5,
    ell_max_diag=18,
):
    disorder = make_fully_critical_disorder(
        L,
        t=t,
        Delta=Delta,
        mu=mu,
        W=W,
        seed=seed,
        bc_x=bc_x,
    )

    U = 2.0 * g

    model = FullyCriticalDisorderedKitaevBDI({
        "L": L,
        "bc_MPS": bc_MPS,
        "bc_x": bc_x,
        "conserve": "parity",
        "mu_array": disorder["mu_array"],
        "t_array": disorder["t_array"],
        "Delta_array": disorder["Delta_array"],
        "U": U,
    })

    psi = MPS.from_product_state(
        model.lat.mps_sites(),
        initial_product_state(L, parity),
        bc=model.lat.bc_MPS,
    )

    dmrg_params = {
        "mixer": mixer,
        "max_sweeps": max_sweeps,
        "combine": True,
        "trunc_params": {
            "chi_max": chi_max,
            "svd_min": svd_min,
        },
        "max_E_err": 1e-10,
        "verbose": verbose,
    }

    engine = dmrg.TwoSiteDMRGEngine(psi, model, dmrg_params)
    E, psi = engine.run()

    n = np.asarray(psi.expectation_value("N"), dtype=float)

    # TeNPy returns entropies for MPS cuts.
    # For the finite MPS representation we keep ell=1,...,L/2.
    S_full = np.asarray(psi.entanglement_entropy(), dtype=float)
    S_half = S_full[: L // 2].copy()

    c_fit, _ = fit_c_from_S(
        S_half,
        L,
        ell_min_diag,
        ell_max_diag,
        bc_x=bc_x,
    )

    cdw_q, nk_abs = cdw_q_from_density(n)

    return DMRGResult(float(E), n, S_half, c_fit, cdw_q, nk_abs)


def run_one_disorder_task(r, params):
    seed = int(params["seed0"]) + int(r)

    try:
        res = run_one_dmrg(
            L=params["L"],
            t=params["t"],
            Delta=params["Delta"],
            mu=params["mu"],
            W=params["W"],
            g=params["g"],
            seed=seed,
            parity=params["parity"],
            chi_max=params["chi_max"],
            max_sweeps=params["max_sweeps"],
            bc_MPS=params["bc_MPS"],
            bc_x=params["bc_x"],
            svd_min=params["svd_min"],
            mixer=params["mixer"],
            verbose=params["verbose_dmrg"],
            ell_min_diag=params["ell_min_diag"],
            ell_max_diag=params["ell_max_diag"],
        )

        return {
            "ok": True,
            "r": int(r),
            "seed": int(seed),
            "E": res.E,
            "S": res.entanglement,
            "n": res.n,
            "cdw_q": res.cdw_q,
            "c_diag": res.central_charge_fit,
            "nk_abs": res.nk_abs,
            "error": "",
            "traceback": "",
        }

    except Exception as err:
        return {
            "ok": False,
            "r": int(r),
            "seed": int(seed),
            "E": np.nan,
            "S": np.full(params["L"] // 2, np.nan),
            "n": np.full(params["L"], np.nan),
            "cdw_q": np.nan,
            "c_diag": np.nan,
            "nk_abs": np.full(params["L"] // 2 + 1, np.nan),
            "error": repr(err),
            "traceback": traceback.format_exc(),
        }


In [8]:
def empty_accumulator():
    return {
        "r_values": np.array([], dtype=int),
        "seeds": np.array([], dtype=int),
        "S_samples": np.empty((0, L // 2), dtype=float),
        "E_samples": np.array([], dtype=float),
        "cdw_samples": np.array([], dtype=float),
        "c_diag_samples": np.array([], dtype=float),
        "ok_flags": np.array([], dtype=bool),
        "errors": np.array([], dtype=object),
    }


def load_accumulator():
    if resume and main_entropy_file.exists():
        data = np.load(main_entropy_file, allow_pickle=True)

        # Prevent mixing OBC and PBC or different L in the same file.
        old_L = int(data["L"]) if "L" in data.files else None
        old_bc_x = str(data["bc_x"]) if "bc_x" in data.files else "unknown"
        old_ell = data["ell_grid"] if "ell_grid" in data.files else None

        compatible = (
            old_L == int(L)
            and old_bc_x == str(bc_x)
            and old_ell is not None
            and len(old_ell) == len(ell_grid)
            and np.all(old_ell == ell_grid)
        )

        if not compatible:
            raise ValueError(
                "Existing save file is incompatible with current run.\n"
                f"File: {main_entropy_file}\n"
                f"saved L={old_L}, current L={L}\n"
                f"saved bc_x={old_bc_x}, current bc_x={bc_x}\n"
                f"saved len(ell_grid)={None if old_ell is None else len(old_ell)}, "
                f"current len(ell_grid)={len(ell_grid)}\n"
                "Use a different file name or set resume=False."
            )

        print("Resuming from", main_entropy_file)

        return {
            "r_values": data["r_values"],
            "seeds": data["seeds"],
            "S_samples": data["S_samples"],
            "E_samples": data["E_samples"],
            "cdw_samples": data["cdw_samples"],
            "c_diag_samples": data["c_diag_samples"],
            "ok_flags": data["ok_flags"],
            "errors": data["errors"] if "errors" in data.files else np.array([], dtype=object),
        }

    return empty_accumulator()


def append_batch(acc, results):
    new = {
        "r_values": np.asarray([o["r"] for o in results], dtype=int),
        "seeds": np.asarray([o["seed"] for o in results], dtype=int),
        "S_samples": np.asarray([o["S"] for o in results], dtype=float),
        "E_samples": np.asarray([o["E"] for o in results], dtype=float),
        "cdw_samples": np.asarray([o["cdw_q"] for o in results], dtype=float),
        "c_diag_samples": np.asarray([o["c_diag"] for o in results], dtype=float),
        "ok_flags": np.asarray([o["ok"] for o in results], dtype=bool),
        "errors": np.asarray([o["error"] for o in results], dtype=object),
    }

    if new["S_samples"].shape[1] != acc["S_samples"].shape[1]:
        raise ValueError(
            "Entropy length mismatch. This usually means that L or boundary condition changed.\n"
            f"old S length = {acc['S_samples'].shape[1]}\n"
            f"new S length = {new['S_samples'].shape[1]}"
        )

    for k in acc:
        if k == "S_samples":
            acc[k] = np.concatenate([acc[k], new[k]], axis=0)
        else:
            acc[k] = np.concatenate([acc[k], new[k]])

    order = np.argsort(acc["r_values"])

    for k in acc:
        acc[k] = acc[k][order]

    return acc


def save_accumulator(acc, batch_index, batch_results=None):
    S_avg = (
        np.nanmean(acc["S_samples"], axis=0)
        if len(acc["S_samples"])
        else np.full(L // 2, np.nan)
    )

    c_avg_diag, s0_avg_diag = fit_c_from_S(
        S_avg,
        L,
        ell_min_diag,
        ell_max_diag,
        bc_x=bc_x,
    )

    np.savez_compressed(
        main_entropy_file,
        ell_grid=ell_grid,
        S_samples=acc["S_samples"],
        S_avg=S_avg,
        r_values=acc["r_values"],
        seeds=acc["seeds"],
        ok_flags=acc["ok_flags"],
        errors=acc["errors"],
        E_samples=acc["E_samples"],
        cdw_samples=acc["cdw_samples"],
        c_diag_samples=acc["c_diag_samples"],
        central_charge_from_avg_S_diag=float(c_avg_diag),
        intercept_from_avg_S_diag=float(s0_avg_diag),
        batch_index=int(batch_index),
        n_done=int(len(acc["r_values"])),
        n_success=int(np.count_nonzero(acc["ok_flags"])),

        # parameters
        L=int(L),
        t=float(t),
        Delta=float(Delta),
        mu=float(mu),
        W=float(W),
        g=float(g),
        chi_max=int(chi_max),
        max_sweeps=int(max_sweeps),
        parity=str(parity),
        bc_MPS=str(bc_MPS),
        bc_x=str(bc_x),
        boundary_tag=str(boundary_tag),
        seed0=int(seed0),
        batch_size=int(batch_size),
        n_jobs=int(n_jobs),
        ell_min_diag=int(ell_min_diag),
        ell_max_diag=int(ell_max_diag),
    )

    if batch_results is not None:
        bf = batch_dir / f"batch_{batch_index:03d}.npz"

        np.savez_compressed(
            bf,
            ell_grid=ell_grid,
            S_batch=np.asarray([o["S"] for o in batch_results], dtype=float),
            r_batch=np.asarray([o["r"] for o in batch_results], dtype=int),
            seed_batch=np.asarray([o["seed"] for o in batch_results], dtype=int),
            ok_batch=np.asarray([o["ok"] for o in batch_results], dtype=bool),
            W=float(W),
            g=float(g),
            L=int(L),
            chi_max=int(chi_max),
            max_sweeps=int(max_sweeps),
            bc_MPS=str(bc_MPS),
            bc_x=str(bc_x),
            boundary_tag=str(boundary_tag),
        )

        print("Saved batch file:", bf)

    print("Saved accumulated file:", main_entropy_file)
    print(
        "n_done =",
        len(acc["r_values"]),
        "n_success =",
        np.count_nonzero(acc["ok_flags"]),
        "c_avg_diag =",
        c_avg_diag,
    )


In [ ]:
params = dict(
    L=L,
    t=t,
    Delta=Delta,
    mu=mu,
    W=W,
    g=g,
    seed0=seed0,
    parity=parity,
    chi_max=chi_max,
    max_sweeps=max_sweeps,
    bc_MPS=bc_MPS,
    bc_x=bc_x,
    svd_min=svd_min,
    mixer=mixer,
    verbose_dmrg=verbose_dmrg,
    ell_min_diag=ell_min_diag,
    ell_max_diag=ell_max_diag,
)

acc = load_accumulator()

# Save an empty file immediately, so you can see the target file before the first batch ends.
if len(acc["r_values"]) == 0 and not main_entropy_file.exists():
    save_accumulator(acc, batch_index=-1, batch_results=None)

r_start = 0 if len(acc["r_values"]) == 0 else int(np.max(acc["r_values"])) + 1

print("Starting from r =", r_start)

for ib in range(n_batches):
    r0 = r_start + ib * batch_size
    r1 = r0 + batch_size
    r_batch = list(range(r0, r1))

    print("\n" + "=" * 80)
    print(f"Batch {ib + 1}/{n_batches}: r={r0},...,{r1 - 1}; n_jobs={min(n_jobs, len(r_batch))}")
    print("=" * 80)

    t0_batch = time.time()

    results = Parallel(
        n_jobs=min(n_jobs, len(r_batch)),
        backend="loky",
        verbose=10,
    )(
        delayed(run_one_disorder_task)(r, params)
        for r in r_batch
    )

    results = sorted(results, key=lambda o: o["r"])

    for o in results:
        if o["ok"]:
            print(
                f"r={o['r']:05d}, seed={o['seed']}, "
                f"E={o['E']:.12g}, "
                f"c_diag={o['c_diag']:.6g}, "
                f"CDW={o['cdw_q']:.6g}"
            )
        else:
            print(f"FAILED r={o['r']:05d}, seed={o['seed']}: {o['error']}")
            print(o["traceback"])

    acc = append_batch(acc, results)
    save_accumulator(acc, batch_index=ib, batch_results=results)

    print(f"Batch runtime: {(time.time() - t0_batch) / 60:.2f} min")

print("Done. Final file:", main_entropy_file)


Resuming from entropy_batches_fully_critical/lifshitz4_entropy_batches_fully_critical_PBC_chi_100.npz
Starting from r = 100

Batch 1/50: r=100,...,119; n_jobs=20


[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.


final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=2.03e-05
/usr/local/lib/python3.12/dist-packages/tenpy/tools/params.py:232: UserWarning: unused option ['verbose'] for config TwoSiteDMRGEngine
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=6.18e-05
/usr/local/lib/python3.12/dist-packages/tenpy/tools/params.py:232: UserWarning: unused option ['verbose'] for config TwoSiteDMRGEngine
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
[Parallel(n_jobs=20)]: Done   2 out of  20 | elapsed: 46.1min remaining: 415.3min
final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=1.57e-05
/usr/local/lib/python3.12/dist-packages/tenpy/tools/params.py:232: UserWarning: unused option ['verbose'] for config TwoSiteDMRGEngine
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err

r=00100, seed=12445, E=-33.7539866466, c_diag=0.290553, CDW=0.0609438
r=00101, seed=12446, E=-32.2859426434, c_diag=0.561427, CDW=0.0511045
r=00102, seed=12447, E=-32.345103645, c_diag=0.219211, CDW=0.0510921
r=00103, seed=12448, E=-33.0868959473, c_diag=1.02055, CDW=0.0566404
r=00104, seed=12449, E=-32.1974983287, c_diag=1.56804, CDW=0.0644634
r=00105, seed=12450, E=-32.8127399661, c_diag=-0.24916, CDW=0.0580929
r=00106, seed=12451, E=-32.8672553884, c_diag=-0.0430429, CDW=0.0623479
r=00107, seed=12452, E=-33.4097248255, c_diag=-0.0400071, CDW=0.0525161
r=00108, seed=12453, E=-33.8327252661, c_diag=0.234507, CDW=0.0521464
r=00109, seed=12454, E=-33.1341137228, c_diag=0.692988, CDW=0.0486633
r=00110, seed=12455, E=-32.3634483334, c_diag=0.517499, CDW=0.0684931
r=00111, seed=12456, E=-33.1081580727, c_diag=0.257044, CDW=0.0625964
r=00112, seed=12457, E=-32.4126355856, c_diag=0.368963, CDW=0.0556723
r=00113, seed=12458, E=-32.7381257416, c_diag=0.793638, CDW=0.0531368
r=00114, seed=12459

/usr/local/lib/python3.12/dist-packages/tenpy/tools/params.py:232: UserWarning: unused option ['verbose'] for config TwoSiteDMRGEngine
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
[Parallel(n_jobs=20)]: Done  20 out of  20 | elapsed: 60.6min finished
[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=3.98e-05
/usr/local/lib/python3.12/dist-packages/tenpy/tools/params.py:232: UserWarning: unused option ['verbose'] for config TwoSiteDMRGEngine
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
/usr/local/lib/python3.12/dist-packages/tenpy/tools/params.py:232: UserWarning: unused option ['verbose'] for config TwoSiteDMRGEngine
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
[Parallel(n_jobs=20)]: Done   2 out of  20 | elapsed: 45.6min remaining: 410.6min
final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=2.31e-05
/usr/l

...

/usr/local/lib/python3.12/dist-packages/tenpy/tools/params.py:232: UserWarning: unused option ['verbose'] for config TwoSiteDMRGEngine
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
[Parallel(n_jobs=20)]: Done  20 out of  20 | elapsed: 58.8min finished
[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=1.63e-05
/usr/local/lib/python3.12/dist-packages/tenpy/tools/params.py:232: UserWarning: unused option ['verbose'] for config TwoSiteDMRGEngine
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
/usr/local/lib/python3.12/dist-packages/tenpy/tools/params.py:232: UserWarning: unused option ['verbose'] for config TwoSiteDMRGEngine
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
[Parallel(n_jobs=20)]: Done   2 out of  20 | elapsed: 46.2min remaining: 415.7min
final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=1.05e-05
/usr/l

r=00220, seed=12565, E=-32.1481597081, c_diag=0.584054, CDW=0.0462223
r=00221, seed=12566, E=-33.1342319773, c_diag=0.0815831, CDW=0.051125
r=00222, seed=12567, E=-33.8801910191, c_diag=0.929314, CDW=0.0518284
r=00223, seed=12568, E=-31.9697837681, c_diag=1.14473, CDW=0.0533257
r=00224, seed=12569, E=-32.8519002878, c_diag=0.370543, CDW=0.0548132
r=00225, seed=12570, E=-32.9152249252, c_diag=0.000931472, CDW=0.0467264
r=00226, seed=12571, E=-32.1686898852, c_diag=1.00604, CDW=0.056135
r=00227, seed=12572, E=-32.385387492, c_diag=0.110925, CDW=0.0825555
r=00228, seed=12573, E=-32.3996940824, c_diag=0.529253, CDW=0.0512807
r=00229, seed=12574, E=-32.3795865376, c_diag=1.09944, CDW=0.0550548
r=00230, seed=12575, E=-32.9699663493, c_diag=0.212655, CDW=0.0534376
r=00231, seed=12576, E=-33.7649811594, c_diag=-0.0202222, CDW=0.053855
r=00232, seed=12577, E=-32.7581592773, c_diag=0.352377, CDW=0.0605128
r=00233, seed=12578, E=-32.7743754878, c_diag=0.160237, CDW=0.0551624
r=00234, seed=12579, 

/usr/local/lib/python3.12/dist-packages/tenpy/tools/params.py:232: UserWarning: unused option ['verbose'] for config TwoSiteDMRGEngine
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
[Parallel(n_jobs=20)]: Done  20 out of  20 | elapsed: 54.4min finished
[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
/usr/local/lib/python3.12/dist-packages/tenpy/tools/params.py:232: UserWarning: unused option ['verbose'] for config TwoSiteDMRGEngine
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=3.40e-05
/usr/local/lib/python3.12/dist-packages/tenpy/tools/params.py:232: UserWarning: unused option ['verbose'] for config TwoSiteDMRGEngine
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
[Parallel(n_jobs=20)]: Done   2 out of  20 | elapsed: 47.3min remaining: 425.4min
final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=8.35e-05
/usr/l

r=00240, seed=12585, E=-33.0987998444, c_diag=0.711064, CDW=0.0560554
r=00241, seed=12586, E=-33.0672736202, c_diag=0.285939, CDW=0.0501941
r=00242, seed=12587, E=-32.9487975798, c_diag=0.236981, CDW=0.0586323
r=00243, seed=12588, E=-32.729467571, c_diag=0.748872, CDW=0.052743
r=00244, seed=12589, E=-33.9484951405, c_diag=0.708417, CDW=0.0622535
r=00245, seed=12590, E=-33.7779302226, c_diag=0.357131, CDW=0.101028
r=00246, seed=12591, E=-32.9149352507, c_diag=0.32313, CDW=0.0510472
r=00247, seed=12592, E=-33.3319483482, c_diag=0.0466578, CDW=0.0636387
r=00248, seed=12593, E=-32.3480643408, c_diag=0.62232, CDW=0.0584324
r=00249, seed=12594, E=-32.9105476421, c_diag=0.735629, CDW=0.0583084
r=00250, seed=12595, E=-32.2464419301, c_diag=0.154059, CDW=0.0488126
r=00251, seed=12596, E=-32.5615105658, c_diag=1.08015, CDW=0.0596776
r=00252, seed=12597, E=-32.9469998473, c_diag=0.838635, CDW=0.0588118
r=00253, seed=12598, E=-32.89640706, c_diag=0.798067, CDW=0.0544269
r=00254, seed=12599, E=-32.

/usr/local/lib/python3.12/dist-packages/tenpy/tools/params.py:232: UserWarning: unused option ['verbose'] for config TwoSiteDMRGEngine
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
[Parallel(n_jobs=20)]: Done  20 out of  20 | elapsed: 57.9min finished
[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=1.36e-05
/usr/local/lib/python3.12/dist-packages/tenpy/tools/params.py:232: UserWarning: unused option ['verbose'] for config TwoSiteDMRGEngine
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
/usr/local/lib/python3.12/dist-packages/tenpy/tools/params.py:232: UserWarning: unused option ['verbose'] for config TwoSiteDMRGEngine
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
[Parallel(n_jobs=20)]: Done   2 out of  20 | elapsed: 46.0min remaining: 413.9min
final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=3.26e-05
/usr/l